# ArXivLens — Attention Visualization Demo

This notebook is the “lens” in **ArXivLens**: it shows *what the cross-encoder
reranker attends to* when deciding whether an arXiv paper is relevant to a query.

**What you'll see:**
- A heatmap of **query tokens × passage tokens** — each cell shows how strongly
  a query token attends to a passage token in the final layer.
- The model runs on **CPU with random weights** by default, so this notebook
  works locally without a GPU or a trained checkpoint. Load a real checkpoint
  (Cell 3 option B) to see meaningful attention patterns from a trained model.

**Architecture reminder:** ArXivLens is a two-stage system:
1. **Dense retrieval** — a bi-encoder (MiniLM) embeds the query and all papers
   independently into a shared vector space and uses FAISS for fast approximate
   nearest-neighbor search. Fast but coarse.
2. **Cross-encoder reranking** — this model. It concatenates the query and passage
   as `[CLS] query [SEP] passage [SEP]` and runs them jointly through a from-scratch
   transformer encoder. Every query token can attend to every passage token and vice
   versa. This joint attention is what makes cross-encoders rank better — and it is
   exactly what this notebook visualizes.

## ⚠️ Attention ≠ Explanation

Before interpreting the heatmap, read this caveat:

> *“Attention is not Explanation”* — Jain & Wallace, NAACL 2019  
> *“Attention is not not Explanation”* — Wiegreffe & Pinter, EMNLP 2019

Attention weights tell you **where the model looks**, not **why it made its
decision**. Two issues:

1. **Attention ≠ importance.** A high attention weight on a token does not
   mean that token causally drove the relevance score. Gradient-based attribution
   methods (e.g., integrated gradients) are more causally grounded than attention.
2. **Multiple heads / layers.** This heatmap averages over all layers and heads.
   Individual heads may encode very different patterns (syntax, coreference,
   proximity) that are lost in the average.

**What the heatmap *is* useful for:** a quick sanity check and a compelling
visual for communicating that the model processes query and passage jointly.
Showing you know this nuance is itself a positive signal to hiring managers.

In [ ]:
"""Cell 3 — Imports and model setup.

Option A (default, works locally): random-weight model for shape demonstration.
Option B (Sol, after training): load a real checkpoint and see learned attention.
"""
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import torch

# Make sure arxivlens is importable when the notebook is run from notebooks/
sys.path.insert(0, str(Path("..") / "src"))

from arxivlens.model.transformer import TransformerConfig
from arxivlens.model.reranker import CrossEncoderReranker

# ── Option A: tiny random-weight model (CPU, no checkpoint needed) ───────────
# d_model=128 and 2 layers keeps forward passes fast on CPU.
# vocab_size=30522 matches bert-base-uncased so we can reuse its tokenizer.
config = TransformerConfig(
    vocab_size=30522,
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=512,
    max_len=256,
    dropout=0.0,   # no dropout during inference
)

# ── Option B: load a trained checkpoint (uncomment on Sol) ───────────────────
# import yaml
# CKPT = Path("/scratch/spate472/mlrag/checkpoints/checkpoint_epoch0004_step006000.pt")
# state = torch.load(CKPT, map_location="cpu")
# cfg = state["config"]["model"]
# config = TransformerConfig(
#     vocab_size=cfg["vocab_size"], d_model=cfg["d_model"], n_heads=cfg["n_heads"],
#     n_layers=cfg["n_layers"],    d_ff=cfg["d_ff"],       max_len=cfg["max_len"],
# )

# ── Tokenizer (borrowed — not built from scratch) ────────────────────────
# The model is from scratch; only the WordPiece tokenizer is borrowed from
# bert-base-uncased, because building a tokenizer is out of scope and not the
# point of this project.
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    print("\u2713 Loaded bert-base-uncased tokenizer")
except Exception as e:
    print(f"\u2717 Could not load tokenizer: {e}")
    print("  Set HF_HOME and ensure bert-base-uncased is cached, or run on Sol.")
    raise

model = CrossEncoderReranker(config, tokenizer=tokenizer, max_length=128)
model.eval()

# ── Option B continued: load weights ──────────────────────────────────────
# model.load_state_dict(state["model_state_dict"])

print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Device           : CPU (move to CUDA with model.to('cuda') on Sol)")

## Define a (query, passage) pair

Edit `QUERY` and `PASSAGE` below to try your own examples.  
The model concatenates them as `[CLS] query [SEP] passage [SEP]` before encoding.

In [ ]:
# ── Query and passage ────────────────────────────────────────────
QUERY = "transformer attention mechanism for natural language processing"

PASSAGE = (
    "We propose a new simple network architecture, the Transformer, based solely "
    "on attention mechanisms, dispensing with recurrence and convolutions entirely. "
    "Experiments on two machine translation tasks show these models to be superior "
    "in quality while being more parallelizable and requiring significantly less "
    "time to train."
)

# ── Run the model ───────────────────────────────────────────────
with torch.no_grad():
    score, attn_info = model.score_with_attention(QUERY, PASSAGE)

print(f"Relevance logit : {score.item():.4f}")
print(f"  (random weights \u2192 meaningless number; train the model for real scores)")
print()

tokens     = attn_info["tokens"]           # list[str] — full joint sequence tokens
sep_index  = attn_info["sep_index"]        # int — index of the first [SEP]
qp_attn    = attn_info["query_passage_attention"]  # (n_layers, n_heads, q_toks, p_toks)

print(f"Joint sequence length : {len(tokens)} tokens")
print(f"First [SEP] at index  : {sep_index}  \u2192 {sep_index - 1} query tokens (excl. [CLS])")
print(f"Attention tensor shape: {tuple(qp_attn.shape)}  (layers, heads, q_toks, p_toks)")

# Query tokens: [CLS]=0, then 1..sep_index-1 are the query tokens
query_tokens   = tokens[1:sep_index]          # exclude [CLS]
passage_tokens = tokens[sep_index + 1:]       # exclude both [SEP]s and padding

# Remove trailing [PAD] tokens from display labels
passage_tokens = [t for t in passage_tokens if t not in ("[PAD]", "[SEP]")]

print(f"Query tokens  ({len(query_tokens)})  : {query_tokens}")
print(f"Passage tokens ({len(passage_tokens)}) : {passage_tokens[:10]} ...")

## Attention Heatmap

The heatmap below shows **query tokens (rows) × passage tokens (columns)**.

Each cell `(i, j)` is the average attention weight that **query token i pays to
passage token j**, averaged over all transformer layers and attention heads.

- **Brighter = more attention.** A bright cell means the query token in that row
  looked hard at the passage token in that column.
- **Row sums are not 1** after averaging across layers/heads (individual head rows
  sum to 1, but the average does not).
- **What to look for:** do the query words “attention” and “transformer” light up
  passage tokens like “attention”, “Transformer”, or “architecture”? With a
  *trained* model, you'd expect topically related tokens to co-attend strongly.
  With random weights, the pattern is noise — which is also worth showing.

In [ ]:
# ── Average attention over layers and heads ──────────────────────────────────
# qp_attn shape: (n_layers, n_heads, q_toks, p_toks)
# Mean over dim 0 (layers) and dim 1 (heads) \u2192 (q_toks, p_toks)
avg_attn = qp_attn.mean(dim=(0, 1)).numpy()   # (n_query_toks, n_passage_toks)

# Clip passage columns to a readable width (first 30 passage tokens)
MAX_PASSAGE_COLS = 30
avg_attn_display   = avg_attn[:, :MAX_PASSAGE_COLS]
display_pass_toks  = passage_tokens[:MAX_PASSAGE_COLS]

# ── Figure ─────────────────────────────────────────────────────────────
fig_width  = max(14, len(display_pass_toks) * 0.45)
fig_height = max(5,  len(query_tokens)      * 0.55)

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

im = ax.imshow(avg_attn_display, aspect="auto", cmap="Blues", vmin=0)

# ── Axis labels ────────────────────────────────────────────────────
ax.set_xticks(range(len(display_pass_toks)))
ax.set_xticklabels(display_pass_toks, rotation=45, ha="right", fontsize=9)

ax.set_yticks(range(len(query_tokens)))
ax.set_yticklabels(query_tokens, fontsize=10)

ax.set_xlabel("Passage tokens  \u2192", fontsize=11)
ax.set_ylabel("Query tokens  \u2192", fontsize=11)
ax.set_title(
    "Cross-Encoder Attention: Query \u00d7 Passage\n"
    f"(averaged over {qp_attn.shape[0]} layers \u00d7 {qp_attn.shape[1]} heads)",
    fontsize=13, pad=12,
)

# ── Colour bar ────────────────────────────────────────────────────
cbar = fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
cbar.set_label("Avg attention weight", fontsize=9)

plt.tight_layout()
plt.savefig("attention_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved to notebooks/attention_heatmap.png")

## Last-layer, last-head attention (single-head view)

The cell above shows the *average*. Below we plot a single head from the last
layer to illustrate how individual heads can encode different patterns.
With a trained model, you'd expect some heads to specialize in semantic matching
(e.g., aligning “attention” in the query to “attention” in the passage) and
others to attend broadly.

In [ ]:
# Last layer, head 0 \u2192 (q_toks, p_toks)
last_layer_head0 = qp_attn[-1, 0].numpy()
last_layer_head0_display = last_layer_head0[:, :MAX_PASSAGE_COLS]

fig2, ax2 = plt.subplots(figsize=(fig_width, fig_height))
im2 = ax2.imshow(last_layer_head0_display, aspect="auto", cmap="Oranges", vmin=0)

ax2.set_xticks(range(len(display_pass_toks)))
ax2.set_xticklabels(display_pass_toks, rotation=45, ha="right", fontsize=9)
ax2.set_yticks(range(len(query_tokens)))
ax2.set_yticklabels(query_tokens, fontsize=10)

ax2.set_xlabel("Passage tokens  \u2192", fontsize=11)
ax2.set_ylabel("Query tokens  \u2192", fontsize=11)
ax2.set_title(
    f"Single-head view: Layer {qp_attn.shape[0]} (last), Head 1\n"
    "(Random weights \u2192 noise. Train the model to see semantic patterns.)",
    fontsize=13, pad=12,
)

cbar2 = fig2.colorbar(im2, ax=ax2, shrink=0.6, pad=0.02)
cbar2.set_label("Attention weight", fontsize=9)

plt.tight_layout()
plt.savefig("attention_heatmap_single_head.png", dpi=150, bbox_inches="tight")
plt.show()
print("Single-head heatmap saved to notebooks/attention_heatmap_single_head.png")

## Next steps

1. **Train the model** on Sol (`sbatch slurm/train_reranker.sh`) using the
   synthetic title\u2192abstract pairs generated by `scripts/build_pairs.py`.
2. **Re-run this notebook** with Option B (load checkpoint) to see attention
   patterns from a model that has actually learned relevance.
3. **Try adversarial examples**: pick a query that shares words with a passage
   that is *not* actually relevant. Does the model still attend to those words?
   Attention patterns in cases where the model gets it *wrong* are often more
   informative than cases where it gets it right.
4. **Plug in the API**: `POST /explain` in `src/arxivlens/serve/api.py` exposes
   the same `score_with_attention` call over HTTP, returning the averaged
   attention matrix as JSON for front-end visualization.